# AISC DeepFake — Calibrated Weighted Score-Level Fusion

Bu notebook, yüklediğin V4 calibrated weighted score-level fusion yönteminin
**senin üç ortak model ailene** uyarlanmış sürümüdür.

Her aile kendi içinde fuse edilir:

- Swin V2 Tiny: Eye + Brow + Mouth
- EfficientNet-B0: Eye + Brow + Mouth
- Swin V2 Tiny + Texture Fusion: Eye + Brow + Mouth

Akış:

`validation regional probabilities -> Platt calibration -> validation ROC-AUC weights -> calibrated TEST probabilities -> weighted fusion`

**TEST seti calibration, weight veya threshold seçimi için kullanılmaz.**
Validation prediction yoksa ilgili aile `BLOCKED_MISSING_VALIDATION` olur.

In [1]:
# ============================================================
# 1) COLAB + PATH CONFIG
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from datetime import datetime, timezone
import json, math, os, re, random, warnings

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

METHOD_NAME = "03_calibrated_weighted_score_fusion"
DECISION_THRESHOLD = 0.50
FRAME_AGGREGATION = "mean"
CALIBRATION_C = 1.0
RELIABILITY_FLOOR = 1e-6
FIGURE_DPI = 600
MIN_FIGURE_SHORT_EDGE_PX = 600

ROOT = Path("/content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1")

RESULT_ROOTS = {
    "eye": ROOT / "Kader/Deney 1/Sonuçlar",
    "brow": ROOT / "Nazlıcan/Deney 1/Sonuçlar",
    "mouth": ROOT / "Dilara/Deney 1/Sonuçlar",
}

OUTPUT_ROOT = RESULT_ROOTS["eye"] / "Fusion_Experiments"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_FAMILIES = {
    "swinv2_tiny": {
        "eye_test": RESULT_ROOTS["eye"] / "20260807_1031_eye_swinv2_tiny_seed42/predictions/test_frame_predictions.csv",
        "brow_test": RESULT_ROOTS["brow"] / "Kas_SwinV2_Tiny_Detayli_Sonuc_pdf/predictions/test_frame_predictions.csv",
        "mouth_test": RESULT_ROOTS["mouth"] / "20260807_2235_mouth_swinv2_tiny_seed42/predictions/test_frame_predictions.csv",
    },
    "efficientnet_b0": {
        "eye_test": RESULT_ROOTS["eye"] / "20260808_0803_eye_efficientnet_b0_seed42/predictions/test_predictions.csv",
        "brow_test": RESULT_ROOTS["brow"] / "20260808_1248_eyebrow_efficientnet_b0_seed42/predictions/test_predictions.csv",
        "mouth_test": RESULT_ROOTS["mouth"] / "20260808_1257_mouth_efficientnet_b0_seed42/predictions/test_predictions.csv",
    },
    "swinv2_texture": {
        "eye_test": RESULT_ROOTS["eye"] / "20260806_1748_eye_swinv2_texturefusion_seed42/full/predictions/test_predictions.csv",
        "brow_test": RESULT_ROOTS["brow"] / "Swin V2-Tiny + LBP + GLCM + Gabor + Wavelet Fusion/predictions/test_predictions_frame_level.csv",
        "mouth_test": RESULT_ROOTS["mouth"] / "SwinV2_TextureFusion_Mouth/20260807_1550_mouth_swinv2_texturefusion_seed42/full/predictions/test_predictions.csv",
    },
}

# True validation prediction CSV varsa buraya yaz.
# TEST CSV kesinlikle yazma.
VALIDATION_OVERRIDE = {
    "swinv2_tiny": {"eye": None, "brow": None, "mouth": None},
    "efficientnet_b0": {"eye": None, "brow": None, "mouth": None},
    "swinv2_texture": {"eye": None, "brow": None, "mouth": None},
}

RUN_ID = datetime.now(timezone.utc).strftime(
    "%Y%m%d_%H%M%S_calibrated_weighted_score_fusion_seed42"
)
RUN_DIR = OUTPUT_ROOT / METHOD_NAME / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)

print("RUN:", RUN_DIR)

Mounted at /content/drive
RUN: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1/Kader/Deney 1/Sonuçlar/Fusion_Experiments/03_calibrated_weighted_score_fusion/20260809_194957_calibrated_weighted_score_fusion_seed42


In [2]:
# ============================================================
# 2) IMPORTS + ATOMIC I/O
# ============================================================

import matplotlib.pyplot as plt
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, confusion_matrix, f1_score, precision_recall_curve,
    precision_score, recall_score, roc_auc_score, roc_curve,
)

def jdefault(x):
    if isinstance(x, (np.integer,)): return int(x)
    if isinstance(x, (np.floating,)): return float(x)
    if isinstance(x, np.ndarray): return x.tolist()
    if isinstance(x, Path): return str(x)
    raise TypeError(type(x).__name__)

def atomic_json(obj, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=jdefault)
        f.flush(); os.fsync(f.fileno())
    with tmp.open("r", encoding="utf-8") as f:
        json.load(f)
    os.replace(tmp, path)

def atomic_csv(df, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, index=False)
    chk = pd.read_csv(tmp)
    if len(chk) != len(df):
        raise RuntimeError(f"CSV verification failed: {path}")
    os.replace(tmp, path)

def save_fig(fig, stem):
    stem = Path(stem); stem.parent.mkdir(parents=True, exist_ok=True)
    png = stem.with_suffix(".png"); svg = stem.with_suffix(".svg")
    tpng = png.with_suffix(".png.tmp"); tsvg = svg.with_suffix(".svg.tmp")
    fig.savefig(tpng, format="png", dpi=FIGURE_DPI, bbox_inches="tight")
    fig.savefig(tsvg, format="svg", bbox_inches="tight")
    with Image.open(tpng) as im:
        if min(im.size) < MIN_FIGURE_SHORT_EDGE_PX:
            raise RuntimeError(f"Low resolution figure: {im.size}")
    os.replace(tpng, png); os.replace(tsvg, svg); plt.close(fig)

In [3]:
# ============================================================
# 3) SCHEMA + FRAME ALIGNMENT
# ============================================================

LABEL_COLS = ["true_label","label","label_int","target","y_true","ground_truth"]
PROB_COLS = ["fake_probability","prob_fake","probability_fake","probability","prob","y_score","score","fake_prob"]
KEY_COLS = ["source_frame","relative_frame_path","frame_path","original_frame","image_path","path","frame_stem"]

def first_col(columns, candidates):
    m = {str(c).lower(): c for c in columns}
    for c in candidates:
        if c.lower() in m: return m[c.lower()]
    return None

def norm_label(v):
    if pd.isna(v): return np.nan
    if isinstance(v, (int,float,np.integer,np.floating)): return int(float(v) >= 0.5)
    s = str(v).strip().lower()
    if s in {"1","fake","deepfake","manipulated","sahte"}: return 1
    if s in {"0","real","genuine","original","gerçek"}: return 0
    return int(float(s) >= 0.5)

def frame_key(v):
    if pd.isna(v): return None
    name = str(v).replace("\\","/").split("/")[-1].lower()
    name = re.sub(r"\.(jpg|jpeg|png|bmp|webp|npy)$","",name)
    m = re.findall(r"(real|fake)_(train|test|val|validation)_(\d+)", name)
    if not m: return None
    label, split, n = m[-1]
    if split == "validation": split = "val"
    return f"{label}_{split}_{str(int(n)).zfill(5)}"

def metadata_candidates(pred_path):
    names = [
        "eligible_metadata.csv","eligible_metadata_before_cache.csv",
        "metadata_used.csv","validated_training_manifest.csv",
        "training_manifest.csv","eligible_mouth_metadata.csv",
    ]
    cur = Path(pred_path).parent
    out = []
    for _ in range(7):
        for n in names:
            out += [cur/"artifacts"/n, cur/"audit"/n, cur/n]
        if cur.parent == cur: break
        cur = cur.parent
    return out

def resolve_keys(raw, pred_path, region):
    for c in KEY_COLS:
        if c in raw.columns:
            keys = raw[c].map(frame_key)
            if float(keys.notna().mean()) >= 0.95:
                return keys, {"key_resolution":"prediction_column","key_column":c}

    if "sample_id" not in raw.columns:
        raise ValueError(f"{region}: no direct frame key and sample_id missing.")

    meta = None; meta_path = None
    for p in metadata_candidates(pred_path):
        if p.is_file():
            try: d = pd.read_csv(p)
            except Exception: continue
            if "sample_id" in d.columns:
                meta, meta_path = d, p
                break
    if meta is None:
        raise FileNotFoundError(f"{region}: companion metadata not found.")

    source_col = None; source_keys = None
    for c in KEY_COLS + ["input_path","input_relative_path","orijinal_yol"]:
        if c in meta.columns:
            k = meta[c].map(frame_key)
            if float(k.notna().mean()) >= 0.95:
                source_col, source_keys = c, k
                break
    if source_col is None:
        raise ValueError(f"{region}: metadata has no trustworthy frame field.")

    mp = pd.DataFrame({
        "sample_id": meta["sample_id"].astype(str).str.strip(),
        "fusion_key": source_keys,
    }).dropna().drop_duplicates("sample_id")

    lookup = pd.DataFrame({
        "sample_id": raw["sample_id"].astype(str).str.strip()
    }).merge(mp, on="sample_id", how="left", validate="many_to_one")

    if lookup["fusion_key"].isna().any():
        raise ValueError(f"{region}: unresolved sample_id values remain.")

    return lookup["fusion_key"], {
        "key_resolution":"companion_metadata",
        "metadata_path":str(meta_path),
        "metadata_source_column":source_col,
    }

def load_region(path, region):
    path = Path(path)
    if not path.is_file(): raise FileNotFoundError(path)
    raw = pd.read_csv(path)
    if raw.empty: raise ValueError(f"{region}: empty CSV")

    lc = first_col(raw.columns, LABEL_COLS)
    pc = first_col(raw.columns, PROB_COLS)
    if lc is None: raise ValueError(f"{region}: label column missing: {list(raw.columns)}")
    if pc is None: raise ValueError(f"{region}: probability column missing: {list(raw.columns)}")

    labels = raw[lc].map(norm_label)
    probs = pd.to_numeric(raw[pc], errors="coerce")
    if labels.isna().any() or probs.isna().any(): raise ValueError(f"{region}: invalid label/probability")
    if ((probs < 0) | (probs > 1)).any(): raise ValueError(f"{region}: probability outside [0,1]")

    keys, audit = resolve_keys(raw, path, region)

    df = pd.DataFrame({
        "fusion_key": keys,
        f"label_{region}": labels.astype(int),
        f"p_{region}": probs.astype(float),
    })

    g = df.groupby("fusion_key", as_index=False).agg(
        **{
            f"label_min_{region}":(f"label_{region}","min"),
            f"label_max_{region}":(f"label_{region}","max"),
            f"p_{region}":(f"p_{region}",FRAME_AGGREGATION),
            f"roi_count_{region}":(f"p_{region}","size"),
        }
    )
    if (g[f"label_min_{region}"] != g[f"label_max_{region}"]).any():
        raise ValueError(f"{region}: conflicting labels inside frame")
    g[f"label_{region}"] = g[f"label_min_{region}"].astype(int)
    g = g.drop(columns=[f"label_min_{region}",f"label_max_{region}"])

    return g, {
        "path":str(path),"source_rows":len(raw),"unique_frames":len(g),
        "label_column":lc,"probability_column":pc,**audit
    }

def align_three(paths):
    eye, ae = load_region(paths["eye"], "eye")
    brow, ab = load_region(paths["brow"], "brow")
    mouth, am = load_region(paths["mouth"], "mouth")

    x = eye.merge(brow,on="fusion_key",how="inner",validate="one_to_one")
    x = x.merge(mouth,on="fusion_key",how="inner",validate="one_to_one")
    if x.empty: raise RuntimeError("Eye/Brow/Mouth common-frame intersection is empty.")

    lm = x[["label_eye","label_brow","label_mouth"]].astype(int)
    if not lm.nunique(axis=1).eq(1).all():
        raise ValueError("Cross-region label disagreement.")
    x["label"] = lm.iloc[:,0].astype(int)
    x = x.sort_values("fusion_key").reset_index(drop=True)

    return x, {
        "counts":{
            "eye_frames":len(eye),"brow_frames":len(brow),"mouth_frames":len(mouth),
            "common_frames":len(x),
            "real_common_frames":int((x["label"]==0).sum()),
            "fake_common_frames":int((x["label"]==1).sum()),
        },
        "eye":ae,"brow":ab,"mouth":am,
    }

In [4]:
# ============================================================
# 4) VALIDATION DISCOVERY + PREFLIGHT
# ============================================================

def local_validation(test_path):
    test_path = Path(test_path)
    dirs = [test_path.parent, test_path.parent.parent/"predictions"]
    if test_path.parent.parent.name.lower() == "full":
        dirs.append(test_path.parent.parent.parent/"predictions")

    candidates = []
    for d in dirs:
        if not d.is_dir(): continue
        for p in d.glob("*.csv"):
            n = p.name.lower()
            if "pred" not in n or "test" in n: continue
            if "validation" in n or re.search(r"(^|[_\-.])val([_\-.]|$)", n):
                candidates.append(p)
    return sorted(set(candidates), key=lambda p:(0 if "frame" in p.name.lower() else 1,len(str(p))))

VALIDATION_PATHS = {}
rows = []

for family, cfg in MODEL_FAMILIES.items():
    VALIDATION_PATHS[family] = {}
    for region in ("eye","brow","mouth"):
        override = VALIDATION_OVERRIDE[family][region]
        if override is not None:
            p = Path(override)
            source = "manual_override"
        else:
            cand = local_validation(cfg[f"{region}_test"])
            p = cand[0] if cand else None
            source = "native_validation_prediction" if p else "not_found"

        if p is not None and not p.is_file():
            p = None; source = "not_found"

        VALIDATION_PATHS[family][region] = p
        rows.append({
            "model_family":family,"region":region,
            "validation_path":str(p) if p else None,"source":source,
        })

validation_discovery = pd.DataFrame(rows)
atomic_csv(validation_discovery, RUN_DIR/"audit"/"validation_prediction_discovery.csv")
display(validation_discovery)

# TEST files must all exist.
test_rows = []
for family,cfg in MODEL_FAMILIES.items():
    for region in ("eye","brow","mouth"):
        p = Path(cfg[f"{region}_test"])
        test_rows.append({"model_family":family,"region":region,"path":str(p),"exists":p.is_file()})

test_audit = pd.DataFrame(test_rows)
display(test_audit)
if not test_audit["exists"].all():
    raise FileNotFoundError("At least one configured TEST prediction file is missing.")

# Family readiness
readiness = []
for family in MODEL_FAMILIES:
    miss = [r for r in ("eye","brow","mouth") if VALIDATION_PATHS[family][r] is None]
    readiness.append({
        "model_family":family,
        "status":"READY" if not miss else "BLOCKED_MISSING_VALIDATION",
        "missing_validation_regions":",".join(miss),
    })

readiness = pd.DataFrame(readiness)
atomic_csv(readiness, RUN_DIR/"audit"/"family_readiness.csv")
display(readiness)

,model_family,region,validation_path,source
0,swinv2_tiny,eye,None,not_found
1,swinv2_tiny,brow,None,not_found
2,swinv2_tiny,mouth,None,not_found
3,efficientnet_b0,eye,None,not_found
4,efficientnet_b0,brow,None,not_found
5,efficientnet_b0,mouth,None,not_found
6,swinv2_texture,eye,None,not_found
7,swinv2_texture,brow,None,not_found
8,swinv2_texture,mouth,None,not_found


,model_family,region,path,exists
0,swinv2_tiny,eye,/content/drive/MyDrive/AISC DeepFake Çalışmala...,True
1,swinv2_tiny,brow,/content/drive/MyDrive/AISC DeepFake Çalışmala...,True
2,swinv2_tiny,mouth,/content/drive/MyDrive/AISC DeepFake Çalışmala...,True
3,efficientnet_b0,eye,/content/drive/MyDrive/AISC DeepFake Çalışmala...,True
4,efficientnet_b0,brow,/content/drive/MyDrive/AISC DeepFake Çalışmala...,True
5,efficientnet_b0,mouth,/content/drive/MyDrive/AISC DeepFake Çalışmala...,True
6,swinv2_texture,eye,/content/drive/MyDrive/AISC DeepFake Çalışmala...,True
7,swinv2_texture,brow,/content/drive/MyDrive/AISC DeepFake Çalışmala...,True
8,swinv2_texture,mouth,/content/drive/MyDrive/AISC DeepFake Çalışmala...,True


,model_family,status,missing_validation_regions
0,swinv2_tiny,BLOCKED_MISSING_VALIDATION,"eye,brow,mouth"
1,efficientnet_b0,BLOCKED_MISSING_VALIDATION,"eye,brow,mouth"
2,swinv2_texture,BLOCKED_MISSING_VALIDATION,"eye,brow,mouth"


In [ ]:
# ============================================================
# 5) CALIBRATION + FUSION
# ============================================================

def fit_calibrator(p, y):
    p = np.asarray(p,dtype=float).reshape(-1,1)
    y = np.asarray(y,dtype=int)
    if len(np.unique(y)) != 2: raise ValueError("Validation must contain both classes.")
    m = LogisticRegression(C=CALIBRATION_C, solver="lbfgs", max_iter=2000, random_state=SEED)
    m.fit(p,y)
    return m

def apply_calibrator(m,p):
    return m.predict_proba(np.asarray(p,dtype=float).reshape(-1,1))[:,1]

def metrics(y,p):
    y = np.asarray(y,dtype=int); p = np.asarray(p,dtype=float)
    pred = (p >= DECISION_THRESHOLD).astype(int)
    tn,fp,fn,tp = confusion_matrix(y,pred,labels=[0,1]).ravel()
    return {
        "n":len(y),"threshold":DECISION_THRESHOLD,
        "accuracy":accuracy_score(y,pred),
        "balanced_accuracy":balanced_accuracy_score(y,pred),
        "precision":precision_score(y,pred,zero_division=0),
        "recall":recall_score(y,pred,zero_division=0),
        "specificity":tn/(tn+fp) if (tn+fp) else np.nan,
        "f1":f1_score(y,pred,zero_division=0),
        "roc_auc":roc_auc_score(y,p),
        "pr_auc":average_precision_score(y,p),
        "brier_score":brier_score_loss(y,p),
        "tn":int(tn),"fp":int(fp),"fn":int(fn),"tp":int(tp),
    }

status_rows=[]; metric_rows=[]; weight_rows=[]

for family,cfg in MODEL_FAMILIES.items():
    print("\n"+"="*90)
    print("MODEL FAMILY:",family)
    print("="*90)

    missing = [r for r in ("eye","brow","mouth") if VALIDATION_PATHS[family][r] is None]
    family_dir = RUN_DIR/family

    if missing:
        rec={"model_family":family,"status":"BLOCKED_MISSING_VALIDATION","missing_regions":",".join(missing)}
        status_rows.append(rec)
        atomic_json(rec,family_dir/"audit"/"BLOCKING_REPORT.json")
        print("BLOCKED:",missing)
        continue

    try:
        val_paths = {r:VALIDATION_PATHS[family][r] for r in ("eye","brow","mouth")}
        test_paths = {r:cfg[f"{r}_test"] for r in ("eye","brow","mouth")}

        val, val_audit = align_three(val_paths)
        test, test_audit = align_three(test_paths)

        overlap = set(val["fusion_key"]) & set(test["fusion_key"])
        if overlap:
            raise RuntimeError(f"Validation/Test frame overlap: {len(overlap)}")

        calibrators={}; aucs={}; rel={}
        for r in ("eye","brow","mouth"):
            cal = fit_calibrator(val[f"p_{r}"],val["label"])
            calibrators[r]=cal
            val[f"cal_p_{r}"] = apply_calibrator(cal,val[f"p_{r}"])
            test[f"cal_p_{r}"] = apply_calibrator(cal,test[f"p_{r}"])
            aucs[r] = float(roc_auc_score(val["label"],val[f"cal_p_{r}"]))
            rel[r] = max(aucs[r]-0.5,RELIABILITY_FLOOR)

        s = sum(rel.values())
        weights = {r:rel[r]/s for r in rel}
        if not math.isclose(sum(weights.values()),1.0,rel_tol=1e-9,abs_tol=1e-9):
            raise RuntimeError("Weights do not sum to 1.")

        for df in (val,test):
            df["fusion_probability"] = sum(weights[r]*df[f"cal_p_{r}"] for r in ("eye","brow","mouth"))

        fam_metrics=[]
        for name,col in {
            "eye_calibrated":"cal_p_eye",
            "brow_calibrated":"cal_p_brow",
            "mouth_calibrated":"cal_p_mouth",
            "fusion":"fusion_probability",
        }.items():
            row={
                "model_family":family,"method":METHOD_NAME,"evaluation":name,
                "calibration_source":"validation_only",
                "weight_source":"validation_roc_auc_reliability",
                "threshold_source":"fixed_0.50",
                "weight_eye":weights["eye"],"weight_brow":weights["brow"],"weight_mouth":weights["mouth"],
                **metrics(test["label"],test[col]),
            }
            fam_metrics.append(row); metric_rows.append(row)

        fam_metrics=pd.DataFrame(fam_metrics)
        atomic_csv(val,family_dir/"predictions"/"aligned_validation_calibrated.csv")
        atomic_csv(test,family_dir/"predictions"/"aligned_test_calibrated_fusion.csv")
        atomic_csv(fam_metrics,family_dir/"metrics"/"test_metrics.csv")

        for r in ("eye","brow","mouth"):
            weight_rows.append({
                "model_family":family,"region":r,"validation_roc_auc":aucs[r],
                "reliability":rel[r],"normalized_weight":weights[r],
                "calibrator_coef":float(calibrators[r].coef_[0,0]),
                "calibrator_intercept":float(calibrators[r].intercept_[0]),
            })

        atomic_json({
            "model_family":family,"method":METHOD_NAME,"weights":weights,
            "validation_auc":aucs,"validation_audit":val_audit,"test_audit":test_audit,
            "test_used_for_calibration":False,"test_used_for_weight_selection":False,
            "test_used_for_threshold_selection":False,
        },family_dir/"audit"/"run_audit.json")

        fm=fam_metrics[fam_metrics["evaluation"]=="fusion"].iloc[0]
        status_rows.append({
            "model_family":family,"status":"SUCCESS",
            "common_validation_frames":len(val),"common_test_frames":len(test),
            "fusion_roc_auc":fm["roc_auc"],"fusion_pr_auc":fm["pr_auc"],"fusion_f1":fm["f1"],
        })
        print(f"SUCCESS | VAL={len(val)} TEST={len(test)} ROC-AUC={fm['roc_auc']:.4f} F1={fm['f1']:.4f}")

    except Exception as exc:
        rec={"model_family":family,"status":"FAILED","error":f"{type(exc).__name__}: {exc}"}
        status_rows.append(rec)
        atomic_json(rec,family_dir/"audit"/"FAILED.json")
        print("FAILED:",type(exc).__name__,exc)

STATUS_DF=pd.DataFrame(status_rows)
METRICS_DF=pd.DataFrame(metric_rows)
WEIGHTS_DF=pd.DataFrame(weight_rows)

atomic_csv(STATUS_DF,RUN_DIR/"family_status.csv")
if not METRICS_DF.empty: atomic_csv(METRICS_DF,RUN_DIR/"metrics"/"all_model_families_metrics.csv")
if not WEIGHTS_DF.empty: atomic_csv(WEIGHTS_DF,RUN_DIR/"metrics"/"calibration_and_weights.csv")

display(STATUS_DF)

In [ ]:
# ============================================================
# 6) FIGURES + FINAL MANIFEST
# ============================================================

for family in MODEL_FAMILIES:
    s = STATUS_DF[STATUS_DF["model_family"]==family]
    if s.empty or s.iloc[0]["status"] != "SUCCESS":
        continue

    fdir = RUN_DIR/family
    test = pd.read_csv(fdir/"predictions"/"aligned_test_calibrated_fusion.csv")
    weights = WEIGHTS_DF[WEIGHTS_DF["model_family"]==family].set_index("region")

    # Weights
    fig,ax=plt.subplots(figsize=(9,6))
    vals=[weights.loc[r,"normalized_weight"] for r in ("eye","brow","mouth")]
    bars=ax.bar(["Eye","Brow","Mouth"],vals)
    ax.set_title(f"{family} — Validation-Derived Region Weights",fontsize=14,fontweight="bold")
    ax.set_ylabel("Normalized Weight"); ax.set_ylim(0,1); ax.grid(axis="y",alpha=.25)
    for b,v in zip(bars,vals):
        ax.text(b.get_x()+b.get_width()/2,b.get_height(),f"{v:.3f}",ha="center",va="bottom")
    fig.tight_layout(); save_fig(fig,fdir/"figures"/"validation_region_weights")

    # ROC
    fig,ax=plt.subplots(figsize=(9,7))
    y=test["label"].to_numpy()
    for name,col in {
        "Eye":"cal_p_eye","Brow":"cal_p_brow","Mouth":"cal_p_mouth","Fusion":"fusion_probability"
    }.items():
        p=test[col].to_numpy(); fpr,tpr,_=roc_curve(y,p); auc=roc_auc_score(y,p)
        ax.plot(fpr,tpr,label=f"{name} (AUC={auc:.3f})",linewidth=2)
    ax.plot([0,1],[0,1],"--",label="Chance")
    ax.set_title(f"{family} — ROC Comparison",fontsize=14,fontweight="bold")
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.legend(); ax.grid(alpha=.25); fig.tight_layout()
    save_fig(fig,fdir/"figures"/"roc_comparison")

    # Confusion matrix
    pred=(test["fusion_probability"].to_numpy()>=DECISION_THRESHOLD).astype(int)
    cm=confusion_matrix(y,pred,labels=[0,1])
    fig,ax=plt.subplots(figsize=(7,6)); im=ax.imshow(cm)
    ax.set_xticks([0,1],labels=["REAL","FAKE"]); ax.set_yticks([0,1],labels=["REAL","FAKE"])
    ax.set_xlabel("Predicted Class"); ax.set_ylabel("True Class")
    ax.set_title(f"{family} — Fusion Confusion Matrix",fontsize=14,fontweight="bold")
    for i in range(2):
        for j in range(2): ax.text(j,i,str(int(cm[i,j])),ha="center",va="center")
    fig.colorbar(im,ax=ax); fig.tight_layout()
    save_fig(fig,fdir/"figures"/"fusion_confusion_matrix")

manifest=[]
for p in sorted(RUN_DIR.rglob("*")):
    if p.is_file():
        rec={"relative_path":str(p.relative_to(RUN_DIR)),"size_bytes":p.stat().st_size}
        if p.suffix.lower()==".png":
            with Image.open(p) as im:
                rec["width_px"]=im.width; rec["height_px"]=im.height
        manifest.append(rec)

atomic_csv(pd.DataFrame(manifest),RUN_DIR/"output_manifest.csv")
atomic_json({
    "run_id":RUN_ID,"method":METHOD_NAME,
    "success_count":int((STATUS_DF["status"]=="SUCCESS").sum()),
    "blocked_count":int((STATUS_DF["status"]=="BLOCKED_MISSING_VALIDATION").sum()),
    "failed_count":int((STATUS_DF["status"]=="FAILED").sum()),
    "test_used_for_calibration":False,
    "test_used_for_weight_selection":False,
    "test_used_for_threshold_selection":False,
},RUN_DIR/"run_summary.json")

print("OUTPUT:",RUN_DIR)
display(STATUS_DF)

## Not

Bu sürüm, V4'teki **calibration + validation reliability weighting + late fusion**
mantığını korur fakat VGG/HOG/GIST/RBF-SVM artifact bağımlılığını kaldırır.

Girdi artık doğrudan senin üç model ailendeki Eye/Brow/Mouth prediction CSV'leridir.
Validation prediction yoksa aile bloklanır; TEST üzerinden calibration yapılmaz.